In [6]:
import pandas as pd
import numpy as np
import nltk

from gensim.models import LdaModel, CoherenceModel
from gensim.corpora.dictionary import Dictionary
from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt

# Lexicon-based sentiment kütüphaneleri
nltk.download("vader_lexicon", quiet=True)
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.corpus import sentiwordnet as swn
from afinn import Afinn

# Diğer NLTK indirmeleri
nltk.download("punkt", quiet=True)
nltk.download('sentiwordnet')
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)

#############################################
# 1. CSV’den Veri Okuma & Ön İşleme
#############################################
csv_path = "C:\\Users\\catsu\\PycharmProjects\\scrappyhotel\\final_processed_reviews_deepl_use.csv"
print("[INFO] CSV okunuyor...")
df = pd.read_csv(csv_path, encoding="utf-8")
df.dropna(subset=["hotel_name", "processed_final_review"], inplace=True)
df.reset_index(drop=True, inplace=True)
reviews = df["processed_final_review"].tolist()
print(f"[INFO] CSV'den {len(reviews)} yorum okundu.")

# Data temiz olduğu için ekstra stopword kaldırımı yapılmıyor.
def tokenize_text(text):
    return nltk.word_tokenize(text.lower())

#############################################
# 2. TF-IDF Keyword Extraction
#############################################
print("\n[INFO] TF-IDF keyword extraction başlatılıyor...")
tfidf_vectorizer = TfidfVectorizer(token_pattern=r'\S+', min_df=2, max_df=0.8)
X_tfidf = tfidf_vectorizer.fit_transform(reviews)
feature_names = np.array(tfidf_vectorizer.get_feature_names_out())
tfidf_sum = np.array(X_tfidf.sum(axis=0)).flatten()
top_n = 10
top_indices = tfidf_sum.argsort()[-top_n:][::-1]
tfidf_keywords = [(feature_names[i], tfidf_sum[i]) for i in top_indices]

print("\n==== TF-IDF Keyword Extraction (Top 10) ====")
for word, score in tfidf_keywords:
    print(f"{word:15} {score:.4f}")

#############################################
# 3. Global LDA Keyword Extraction
#############################################
print("\n[INFO] Global LDA keyword extraction başlatılıyor...")
tokenized_texts = [tokenize_text(text) for text in reviews]
# Sözlük ve corpus oluşturuluyor; çok nadir veya çok sık geçen kelimeler filtreleniyor.
dictionary = Dictionary(tokenized_texts)
dictionary.filter_extremes(no_below=2, no_above=0.8)
corpus = [dictionary.doc2bow(t) for t in tokenized_texts]

# Global LDA: Optimal topic sayısını aramadan sabit 3 topic kullanıyoruz.
num_topics = 3
lda_model = LdaModel(corpus=corpus, id2word=dictionary, num_topics=num_topics, passes=15, random_state=42)
def extract_lda_keywords(model, topic_id, top_n=10):
    terms = model.show_topic(topic_id, topn=top_n)
    return [(term, weight) for term, weight in terms]

lda_keywords = {}
print("\n==== LDA Keyword Extraction (Top 10 per topic) ====")
for t in range(num_topics):
    lda_keywords[t] = extract_lda_keywords(lda_model, t, top_n=10)
    print(f"\nTopic {t}:")
    for term, weight in lda_keywords[t]:
        print(f"  {term:15} {weight:.4f}")

#############################################
# 4. Lexicon-Based Sentiment Analysis
#############################################
print("\n[INFO] Lexicon-based sentiment analizi başlatılıyor...")

# VADER
sia = SentimentIntensityAnalyzer()
def get_vader_sentiment(text):
    scores = sia.polarity_scores(text)
    compound = scores["compound"]
    if compound >= 0.05:
        label = "positive"
    elif compound <= -0.05:
        label = "negative"
    else:
        label = "neutral"
    return compound, label

# SentiWordNet
def get_sentiwordnet_sentiment(word):
    synsets = list(swn.senti_synsets(word))
    if not synsets:
        return None
    synset = synsets[0]
    return {
        "positivity": synset.pos_score(),
        "negativity": synset.neg_score(),
        "objectivity": synset.obj_score()
    }
def analyze_text_sentiwordnet(text):
    words = nltk.word_tokenize(text)
    # Data temiz olduğu için stopword filtresi eklenmiyor.
    total = {"positivity": 0.0, "negativity": 0.0, "objectivity": 0.0}
    for word in words:
        if word.isalpha():
            scores = get_sentiwordnet_sentiment(word.lower())
            if scores:
                total["positivity"] += scores["positivity"]
                total["negativity"] += scores["negativity"]
                total["objectivity"] += scores["objectivity"]
    return total

# AFINN
afinn = Afinn()
def get_afinn_sentiment(text):
    return afinn.score(text)

#############################################
# 5. Her Yorum için Lexicon Sonuçları ve CSV Yazımı
#############################################
print("\n[INFO] Lexicon-based sentiment analizi her yorum için uygulanıyor...")
lexicon_results = []
for i, text in enumerate(reviews):
    vader_score, vader_label = get_vader_sentiment(text)
    senti_scores = analyze_text_sentiwordnet(text)
    afinn_score = get_afinn_sentiment(text)
    lexicon_results.append({
        "hotel_name": df.loc[i, "hotel_name"],
        "review_text": text,
        "vader_compound": vader_score,
        "vader_label": vader_label,
        "sentiwordnet": senti_scores,
        "afinn_score": afinn_score
    })
    if i % 100 == 0:
        print(f"[INFO] {i} yorum işlendi...")

lexicon_df = pd.DataFrame(lexicon_results)
sentiment_csv = "lexicon_sentiment_summary.csv"
lexicon_df.to_csv(sentiment_csv, index=False, encoding="utf-8-sig")
print(f"\n[INFO] Lexicon-based sentiment sonuçları '{sentiment_csv}' dosyasına kaydedildi.")

#############################################
# 6. Özet CSV: Keyword Extraction Sonuçları
#############################################
print("\n[INFO] Keyword extraction özet CSV oluşturuluyor...")
summary = {
    "tfidf_top_keywords": "; ".join([f"{w}({score:.2f})" for w, score in tfidf_keywords])
}
for t in range(num_topics):
    summary[f"lda_topic_{t}_keywords"] = "; ".join([f"{w}({weight:.3f})" for w, weight in lda_keywords[t]])
summary_df = pd.DataFrame([summary])
keywords_csv = "keyword_extraction_summary.csv"
summary_df.to_csv(keywords_csv, index=False, encoding="utf-8-sig")
print(f"[INFO] Keyword extraction sonuçları '{keywords_csv}' dosyasına kaydedildi.")

print("\n***** Tüm işlemler tamamlandı! *****")


[nltk_data] Downloading package sentiwordnet to
[nltk_data]     C:\Users\catsu\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\sentiwordnet.zip.


[INFO] CSV okunuyor...
[INFO] CSV'den 1271 yorum okundu.

[INFO] TF-IDF keyword extraction başlatılıyor...

==== TF-IDF Keyword Extraction (Top 10) ====
room            74.1013
hotel           70.6177
stay            56.3090
clean           54.3570
place           51.3048
nice            43.4097
breakfast       41.2904
location        38.6208
good            35.7431
akyaka          35.5153

[INFO] Global LDA keyword extraction başlatılıyor...

==== LDA Keyword Extraction (Top 10 per topic) ====

Topic 0:
  breakfast       0.0257
  hotel           0.0247
  room            0.0245
  clean           0.0112
  stay            0.0091
  good            0.0085
  place           0.0081
  location        0.0077
  akyaka          0.0069
  price           0.0067

Topic 1:
  room            0.0396
  hotel           0.0300
  stay            0.0229
  clean           0.0206
  place           0.0149
  location        0.0127
  nice            0.0115
  friendly        0.0103
  akyaka          0.0102
  tha